<img src='http://hilpisch.com/tpq_logo.png' width="350px" align="left">
<img src='http://hilpisch.com/taim_logo.png' width="300px" align="right">

# Oanda Master Class

## Event-Based Backtesting

Dr Yves J Hilpisch | The Python Quants | The AI Machine

<td><a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://aimachine.io" target="_blank">http://aimachine.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:team@tpq.io">team@tpq.io</a></td>

## Imports

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_addon.git
import sys
sys.path.append('python_for_algo_trading_addon')


In [ ]:
!pip install git+https://github.com/yhilpisch/tpqoa

In [ ]:
import numpy as np
import pandas as pd
from pylab import plt
plt.style.use('seaborn-v0_8')

## API Connection

In [ ]:
import tpqoa

In [ ]:
api = tpqoa.tpqoa('/content/python_for_algo_trading_addon/pyalgo.cfg')

## Financial Data Class

In [ ]:
instrument = 'EUR_USD'
start = '2020-03-17'
end = '2020-03-19'
granularity = 'M10'
price = 'M'

In [ ]:
class OandaData:
    def __init__(self, instrument, start, end, granularity, price):
        self.instrument = instrument
        self.start = start
        self.end = end
        self.granularity = granularity
        self.price = price
        self._retrieve_data()
        self._prepare_data()
    def _retrieve_data(self):
        self.raw = api.get_history(self.instrument,
                    self.start, self.end, self.granularity,
                    self.price, localize=True)
    def _prepare_data(self):
        self.data = pd.DataFrame(self.raw['c'])
        self.data.columns = [self.instrument]
        self.data['r'] = np.log(self.data / self.data.shift(1))
        self.data.dropna(inplace=True)
    def plot_data(self, cols=None):
        if cols is None:
            cols = [self.instrument]
        self.data[cols].plot(figsize=(10, 6),
                title=f'{self.instrument} | {self.granularity}')

In [ ]:
od = OandaData(instrument, start, end, granularity, price)

In [ ]:
od.raw.info()

In [ ]:
od.data.head()

In [ ]:
od.plot_data()

## Backtesting Base

In [ ]:
class BacktestingBase(OandaData):
    def __init__(self, instrument, start, end, granularity, price,
                amount, verbose=True):
        super(BacktestingBase, self).__init__(instrument, start, end,
                                              granularity, price)
        self.initial_amount = amount
        self.current_balance = amount
        self.units = 0
        self.trades = 0
        self.verbose = verbose
    def get_date_price(self, bar):
        date = str(self.data.index[bar])
        price = self.data[self.instrument].iloc[bar]
        return date, price
    def print_current_balance(self, bar):
        date, price = self.get_date_price(bar)
        print(f'{date} | current balance = {self.current_balance:.2f}')
    def print_net_wealth(self, bar):
        date, price = self.get_date_price(bar)
        net_wealth = self.current_balance + self.units * price
        print(f'{date} | net_wealth = {net_wealth:.2f}')
    def report_trade(self, side, bar, units):
        date, price = self.get_date_price(bar)
        print(60 * '=')
        print(f'{date} | *** PLACING {side} ORDER ***')
        print(f'{date} | units={units} | price={price:.4f}')
        self.print_current_balance(bar)
        self.print_net_wealth(bar)
        print(60 * '=')
    def place_buy_order(self, bar, units=None, amount=None):
        date, price = self.get_date_price(bar)
        if units is None:
            units = int(amount / price)
        self.current_balance -= units * price
        self.units += units
        if self.verbose:
            self.report_trade('BUY', bar, units)
        self.trades += 1
    def place_sell_order(self, bar, units=None, amount=None):
        date, price = self.get_date_price(bar)
        if units is None:
            units = int(amount / price)
        self.current_balance += units * price
        self.units -= units
        if self.verbose:
            self.report_trade('SELL', bar, units)
        self.trades += 1
    def close_out(self, bar):
        date, price = self.get_date_price(bar)
        print(60 * '=')
        print(f'{date} | *** CLOSING POSITION ***')
        if self.units >= 0:
            self.place_sell_order(bar, units=self.units)
        else:
            self.place_buy_order(bar, units=-self.units)
        print(60 * '=')
        perf = (self.current_balance - self.initial_amount) / self.initial_amount
        print(f'{date} | net performance = {perf:.4f}')
        print(f'{date} | number of trades = {self.trades}')
        print(60 * '=')

In [ ]:
bb = BacktestingBase(instrument, start, end,
                     granularity, price, 10000,
                     verbose=True)

In [ ]:
bar = 100

In [ ]:
bb.get_date_price(bar)

In [ ]:
bb.print_current_balance(bar)

In [ ]:
bb.place_buy_order(bar, units=5000)

In [ ]:
bb.print_current_balance(bar)

In [ ]:
bb.print_net_wealth(bar)

In [ ]:
bb.place_sell_order(2 * bar, units = 2500)

In [ ]:
bb.print_current_balance(2 * bar)

In [ ]:
bb.print_net_wealth(2 * bar)

In [ ]:
bb.close_out(2 * bar + 25)

## Backtesting Class

In [ ]:
class SMABacktesting(BacktestingBase):
    def _prepare_indicators(self):
        self.data['SMA1'] = self.data[self.instrument].rolling(self.SMA1).mean()
        self.data['SMA2'] = self.data[self.instrument].rolling(self.SMA2).mean()
        self.data['p'] = np.where(self.data['SMA1'] > self.data['SMA2'],
                                  'long', 'short')
    def backtest_strategy(self, SMA1, SMA2, units):
        self.SMA1 = SMA1
        self.SMA2 = SMA2
        self.units = 0
        self.trades = 0
        self.position = 0
        self.current_balance = self.initial_amount
        self._prepare_indicators()
        for bar in range(SMA2, len(self.data)):
            if self.position in [0, -1]:
                if self.data['p'].iloc[bar] == 'long':
                    self.place_buy_order(bar,
                        units=(1 - self.position) * units)
                    self.position = 1
            elif self.position in [0, 1]:
                if self.data['p'].iloc[bar] == 'short':
                    self.place_sell_order(bar,
                        units=(1 + self.position) * units)
                    self.position = -1
        self.close_out(bar)

In [ ]:
sma = SMABacktesting(instrument, start, end,
                     granularity, price, 10000,
                     verbose=True)

In [ ]:
sma.backtest_strategy(10, 35, 100000)

<img src='http://hilpisch.com/tpq_logo.png' width="350px" align="left">
<img src='http://hilpisch.com/taim_logo.png' width="300px" align="right">